# 10주차 1팀
# 1. 9주차 내용

In [7]:
# 1) 내 Google Drive 연결 — 원시자료는 각자 자기 Drive 에 둡니다
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pyreadstat
import pyreadstat
DATA_DIR = '/content/drive/MyDrive/hdac_data' # 팀 공통 경로

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# ==============================================================================
# Google Drive 연결 및 SAS 원시자료 로드 (실제 변수명 반영)
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')

!pip -q install pyreadstat
import pyreadstat
import pandas as pd
import numpy as np

# 팀 공통 경로 및 파일명 설정
DATA_DIR = '/content/drive/MyDrive/원시자료'
FILE_NAME = 'kyrbs2025.sas7bdat'

# 분석에 사용할 실제 원변수명 목록
cols = [
    'M_SLP_HR_K',   # 주중 잠든 시각 (시간) - 노출 산출용
    'M_SLP_MM_K',   # 주중 잠든 시각 (분) - 노출 산출용
    'M_WK_HR_K',    # 주중 일어난 시각 (시간) - 노출 산출용
    'M_WK_MM_K',    # 주중 일어난 시각 (분) - 노출 산출용
    'E_S_RCRD',     # 주관적 학업성적 (결과)
    'SEX',          # 성별 (공변량)
    'GRADE',        # 학년 (공변량)
    'E_SES',        # 가구 경제 상태 (공변량)
    'M_STR',        # 평상시 스트레스 인지 (공변량)
    'PA_SWD_S',     # 주중 학습 목적 앉아서 보낸 시간 (공변량)
    'F_BR'          # 아침식사 횟수 등 (공변량)
]

print("SAS 원시자료를 불러오는 중입니다...")

# 파일 경로 조합 및 데이터 로드
file_path = f'{DATA_DIR}/{FILE_NAME}'
df, meta = pyreadstat.read_sas7bdat(file_path, usecols=cols)

print(f"[로드 완료] 전체 데이터 형태(행, 열): {df.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SAS 원시자료를 불러오는 중입니다...
[로드 완료] 전체 데이터 형태(행, 열): (54170, 11)


In [9]:
# =========================================================
# 2. 변수 확인 및 원 코드 빈도표
# =========================================================
print("=" * 60 + "\n[2] 변수 확인 및 원 코드 빈도표\n" + "=" * 60)

# 설계 변수 후보 자동 탐색 및 결합 (안전 장치)
WEIGHT_CANDS  = ['W', 'w']
STRATA_CANDS  = ['STRATA', 'strata']
CLUSTER_CANDS = ['CLUSTER', 'cluster', 'PSU', 'psu']

_, meta_all = pyreadstat.read_sas7bdat(file_path, metadataonly=True)
all_cols = list(meta_all.column_names)

def find_var(cands):
    return next((c for c in cands if c in all_cols), None)

W_VAR       = find_var(WEIGHT_CANDS)
STRATA_VAR  = find_var(STRATA_CANDS)
CLUSTER_VAR = find_var(CLUSTER_CANDS)
design_vars = [v for v in [W_VAR, STRATA_VAR, CLUSTER_VAR] if v]

if design_vars and not all(v in df.columns for v in design_vars):
    df_design, _ = pyreadstat.read_sas7bdat(file_path, usecols=design_vars)
    df_raw = pd.concat([df.reset_index(drop=True), df_design.reset_index(drop=True)], axis=1)
else:
    df_raw = df.copy()

print(f"원시자료 행 수: {len(df_raw):,}, 읽은 변수 수: {df_raw.shape[1]}")
print(f"설계변수 확인 -> 가중치={W_VAR}, 층={STRATA_VAR}, 집락={CLUSTER_VAR}")

summary = pd.DataFrame({
    "dtype": df_raw[df_raw.columns].dtypes.astype(str),
    "n_nan": df_raw[df_raw.columns].isna().sum(),
    "n_unique": df_raw[df_raw.columns].nunique(),
    "min": df_raw[df_raw.columns].min(numeric_only=True),
    "max": df_raw[df_raw.columns].max(numeric_only=True),
})
display(summary)

def freq_table(s):
    vc = s.value_counts(dropna=False).sort_index()
    out = pd.DataFrame({"n": vc, "%": (vc / vc.sum() * 100).round(2)})
    out.index.name = s.name
    return out

print("\n--- 주요 공변량 원 코드 빈도표 ---")
check_vars = ['E_S_RCRD', 'SEX', 'GRADE', 'E_SES', 'M_STR', 'F_BR']
for v in check_vars:
    if v in df_raw.columns:
        print(f"\n[변수: {v}]")
        display(freq_table(df_raw[v]))

print("\n[PA_SWD_S (주중 학습목적 앉아있는 시간 - 분 단위) 요약통계]")
display(df_raw['PA_SWD_S'].describe())

[2] 변수 확인 및 원 코드 빈도표
원시자료 행 수: 54,170, 읽은 변수 수: 14
설계변수 확인 -> 가중치=W, 층=STRATA, 집락=CLUSTER


,dtype,n_nan,n_unique,min,max
CLUSTER,float64,0,796,1.000000,800.0
E_SES,float64,12,5,1.000000,5.0
E_S_RCRD,float64,6,5,1.000000,5.0
F_BR,float64,0,8,1.000000,8.0
GRADE,float64,0,6,1.000000,6.0
M_SLP_HR_K,float64,0,24,1.000000,24.0
M_SLP_MM_K,float64,0,6,0.000000,50.0
M_STR,float64,0,5,1.000000,5.0
M_WK_HR_K,float64,0,24,1.000000,24.0
M_WK_MM_K,float64,0,6,0.000000,50.0



--- 주요 공변량 원 코드 빈도표 ---

[변수: E_S_RCRD]


,n,%
E_S_RCRD,,
1.0,6478,11.96
2.0,13870,25.60
3.0,15669,28.93
4.0,12865,23.75
5.0,5282,9.75
NaN,6,0.01



[변수: SEX]


,n,%
SEX,,
1.0,27761,51.25
2.0,26409,48.75


... (6 outputs left out)


[변수: F_BR]


,n,%
F_BR,,
1.0,14308,26.41
2.0,4236,7.82
3.0,5105,9.42
4.0,4153,7.67
5.0,3266,6.03
6.0,5859,10.82
7.0,3306,6.10
8.0,13937,25.73



[PA_SWD_S (주중 학습목적 앉아있는 시간 - 분 단위) 요약통계]


,PA_SWD_S
count,53467.000000
mean,432.751305
std,244.829807
min,0.000000
25%,240.000000
50%,450.000000
75%,600.000000
max,1438.000000


In [10]:
# =========================================================
# 3. 무응답·비해당 코드 결측 처리 및 수면시간 산출
# =========================================================
print("=" * 60)
print("[3] 무응답·비해당 코드 결측 처리 및 수면시간 산출")
print("=" * 60)

d = df_raw.copy()

# 1) 취침/기상 시각의 24시 응답을 0시(자정)로 변환
d.loc[d['M_SLP_HR_K'] == 24, 'M_SLP_HR_K'] = 0
d.loc[d['M_WK_HR_K'] == 24, 'M_WK_HR_K'] = 0

# 2) 이용지침서 기준 유효값 범위 설정 (PA_SWD_S 연속형 0~1438분 반영)
VALID = {
    'M_SLP_HR_K': range(0, 24),   # 0~23시 (24시는 0시로 변환됨)
    'M_SLP_MM_K': range(0, 60),
    'M_WK_HR_K':  range(0, 24),
    'M_WK_MM_K':  range(0, 60),
    'E_S_RCRD':   range(1, 6),    # 1(상) ~ 5(하)
    'SEX':        [1, 2],         # 1(남), 2(여)
    'GRADE':      range(1, 7),    # 1~3(중1~3), 4~6(고1~3)
    'E_SES':      range(1, 6),    # 1(상) ~ 5(하)
    'M_STR':      range(1, 6),
    'PA_SWD_S':   range(0, 1439), # 분 단위 연속형 (0~1,438분)
    'F_BR':       range(1, 9),    # 아침식사 횟수 (1~8)
}

# 유효범위를 벗어난 응답을 결측(NaN)으로 처리
for v, rng in VALID.items():
    if v in d.columns:
        d.loc[~d[v].isin(list(rng)), v] = np.nan

# 3) 수면시간 산출 (분 단위 -> 시간 단위 변환, 자정 넘김 보정)
bed_min = d['M_SLP_HR_K'] * 60 + d['M_SLP_MM_K']
wake_min = d['M_WK_HR_K'] * 60 + d['M_WK_MM_K']
sleep_min = np.where(wake_min < bed_min, wake_min + 1440 - bed_min, wake_min - bed_min)
d['SLEEP_HR'] = sleep_min / 60.0

# 4) 비현실적 수면시간 이상값 처리 (2시간 미만 또는 18시간 초과)
d.loc[(d['SLEEP_HR'] < 2.0) | (d['SLEEP_HR'] > 18.0), 'SLEEP_HR'] = np.nan

# 5) 노출군(EXPOSURE) 정의 (주 분석을 위한 2집단 체계로 확정)
# 0: 권장 수면충족군 (8~10시간, 기준군)
# 1: 수면 부족군 (< 8시간, 노출군)
# 10시간 초과 군은 주 분석 대상에서 제외(NaN 처리)하여 연구질문과 일치시킴
d['EXPOSURE'] = np.nan
d.loc[(d['SLEEP_HR'] >= 8.0) & (d['SLEEP_HR'] <= 10.0), 'EXPOSURE'] = 0
d.loc[d['SLEEP_HR'] < 8.0, 'EXPOSURE'] = 1

print("수면시간(SLEEP_HR) 요약통계:")
display(d['SLEEP_HR'].describe())

def freq_table(s):
    vc = s.value_counts(dropna=False).sort_index()
    out = pd.DataFrame({"n": vc, "%": (vc / vc.sum() * 100).round(2)})
    out.index.name = s.name
    return out

print()
print("확정된 노출군(EXPOSURE: 0=8~10시간, 1=<8시간) 분포 빈도:")
display(freq_table(d['EXPOSURE']))

[3] 무응답·비해당 코드 결측 처리 및 수면시간 산출
수면시간(SLEEP_HR) 요약통계:


,SLEEP_HR
count,50783.000000
mean,8.206434
std,2.024602
min,2.000000
25%,7.000000
50%,8.000000
75%,9.333333
max,18.000000



확정된 노출군(EXPOSURE: 0=8~10시간, 1=<8시간) 분포 빈도:


,n,%
EXPOSURE,,
0.0,23402,43.20
1.0,20741,38.29
NaN,10027,18.51


In [11]:
import pandas as pd

print("=" * 60)
print("[4] 대상 선정 단계별 인원 및 공변량 결측 분석")
print("=" * 60)

steps = []
def log_step(step_name, criterion, x):
    n_curr = len(x)
    n_prev = steps[-1]["남은 인원(명)"] if steps else len(d)
    steps.append({
        "단계": step_name,
        "기준": criterion,
        "남은 인원(명)": n_curr,
        "단계 제외(명)": n_prev - n_curr,
        "누적 제외율(%)": round(((len(d) - n_curr) / len(d)) * 100, 2),
        "가중합(추정모집단)": round(x[W_VAR].sum(), 0) if 'W_VAR' in globals() and W_VAR and W_VAR in x.columns else None
    })

# 1단계: 전체 원시자료 응답자
s1 = d.copy()
log_step("1단계", "전체 원시자료 응답자", s1)

# 2단계: 모집단 필터링 (중1~고3 재학생)
s2 = s1[s1['GRADE'].between(1, 6)].copy()
log_step("2단계", "중1~고3 재학생 (GRADE 1~6)", s2)

# 3단계: 노출 변수(수면시간) 유효자 선정
s3 = s2.dropna(subset=['EXPOSURE']).copy()
log_step("3단계", "주중 수면시간 유효자 (권장충족군 및 부족군)", s3)

# 4단계: 결과 변수(학업성적) 결측 제외 -> Table 1 및 기본 분석 대상자 (예: 44,138명)
s4 = s3.dropna(subset=['E_S_RCRD']).copy()
log_step("4단계", "주관적 학업성적 유효 응답자 (Table 1 및 기본 분석 대상)", s4)

flow_df = pd.DataFrame(steps)
display(flow_df)

# Table 1 기본 분석 대상자 기준 공변량별 결측 분석
COVARS = ['SEX', 'GRADE', 'E_SES', 'M_STR', 'PA_SWD_S', 'F_BR']
print()
print(f"[Table 1 기본 분석 대상자(N = {len(s4):,}) 내 공변량별 결측 현황]")
covar_missing = pd.DataFrame({
    "유효 응답자 수": s4[COVARS].notna().sum(),
    "결측자 수": s4[COVARS].isna().sum(),
    "결측 비율(%)": (s4[COVARS].isna().sum() / len(s4) * 100).round(2)
})
display(covar_missing)

# 5단계: 회귀분석을 위한 주요 공변량 결측 최종 제외 (완전 분석 대상, 예: 43,562명)
# ※ 선정 단계 흐름(Flow chart)에는 포함하지 않고, 회귀분석 모델링 직전에만 적용합니다.
df_regression = s4.dropna(subset=COVARS).copy()

print()
print(f"최종 다중변수 회귀분석 대상자 수: N = {len(df_regression):,}명")
print("회귀분석 대상 노출군 분포:")
display(freq_table(df_regression['EXPOSURE']))

[4] 대상 선정 단계별 인원 및 공변량 결측 분석


,단계,기준,남은 인원(명),단계 제외(명),누적 제외율(%),가중합(추정모집단)
0,1단계,전체 원시자료 응답자,54170,0,0.00,2642704.0
1,2단계,중1~고3 재학생 (GRADE 1~6),54170,0,0.00,2642704.0
2,3단계,주중 수면시간 유효자 (권장충족군 및 부족군),44143,10027,18.51,2166085.0
3,4단계,주관적 학업성적 유효 응답자 (Table 1 및 기본 분석 대상),44138,5,18.52,2165847.0



[Table 1 기본 분석 대상자(N = 44,138) 내 공변량별 결측 현황]


,유효 응답자 수,결측자 수,결측 비율(%)
SEX,44138,0,0.00
GRADE,44138,0,0.00
E_SES,44134,4,0.01
M_STR,44138,0,0.00
PA_SWD_S,43565,573,1.30
F_BR,44138,0,0.00



최종 다중변수 회귀분석 대상자 수: N = 43,562명
회귀분석 대상 노출군 분포:


,n,%
EXPOSURE,,
0.0,23098,53.02
1.0,20464,46.98


# 2. [B-2] Table 1

In [12]:
import pandas as pd
import numpy as np

# =========================================================
# [Table 1] 교수님 피드백 완벽 반영 Table 1 생성 코드
# 데이터셋: s4 (N = 44,138, 주관적 학업성적 및 수면시간 유효자)
# =========================================================

# 1. 집단 구분 및 샘플 수 (비가중 N)
n_total = len(s4)
n_ref = (s4['EXPOSURE'] == 0.0).sum()   # 권장 수면군 (8~10시간)
n_exp = (s4['EXPOSURE'] == 1.0).sum()   # 수면 부족군 (<8시간)

print(f"Table 1 기본 대상자 수: 전체 N = {n_total:,} | 권장수면군 N = {n_ref:,} | 수면부족군 N = {n_exp:,}\n")

# 복합표본 가중 평균 및 Design SE 산출 함수 (PA_SWD_S 등 연속형용)
def get_complex_survey_mean_se(df, var_name):
    # 가중 평균 (Weighted Mean)
    w_mean = np.average(df[var_name], weights=df['W'])

    # 층(STRATA) 및 집락(CLUSTER) 반영 선형화 분산(Taylor Series Linearization)을 통한 Design SE 계산
    # 각 집락(PSU)별 잔차 가중합 산출
    df_temp = df[[var_name, 'W', 'STRATA', 'CLUSTER']].copy()
    df_temp['dev'] = (df_temp[var_name] - w_mean) * df_temp['W']

    psu_sum = df_temp.groupby(['STRATA', 'CLUSTER'])['dev'].sum().reset_index()
    strata_means = psu_sum.groupby('STRATA')['dev'].mean().reset_index()

    merged = pd.merge(psu_sum, strata_means, on='STRATA', suffixes=('', '_strata_mean'))
    merged['diff_sq'] = (merged['dev'] - merged['dev_strata_mean']) ** 2

    # 층별 집락 수(n_h) 반영 variance 계산
    strata_nh = psu_sum.groupby('STRATA')['CLUSTER'].nunique().reset_index(name='n_h')
    strata_var = pd.merge(merged.groupby('STRATA')['diff_sq'].sum().reset_index(), strata_nh, on='STRATA')

    # Single PSU strata 처리 (n_h > 1 인 경우 적용)
    strata_var = strata_var[strata_var['n_h'] > 1]
    strata_var['var_h'] = (strata_var['n_h'] / (strata_var['n_h'] - 1)) * strata_var['diff_sq']

    total_var = strata_var['var_h'].sum()
    total_weight_sq = (df['W'].sum()) ** 2

    design_se = np.sqrt(total_var) / df['W'].sum()
    return w_mean, design_se

# ---------------------------------------------------------
# 2. Table 1 데이터 프레임 생성 로직
# ---------------------------------------------------------
table1_rows = []

# 분석 대상 변수 목록 (범주형/연속형 구분)
cat_vars = ['SEX', 'GRADE', 'E_SES', 'M_STR', 'F_BR', 'E_S_RCRD']
cont_vars = ['PA_SWD_S']

# 라벨 맵핑
labels = {
    'SEX': {1: '남성', 2: '여성'},
    'GRADE': {1: '중학교 1학년', 2: '중학교 2학년', 3: '중학교 3학년', 4: '고등학교 1학년', 5: '고등학교 2학년', 6: '고등학교 3학년'},
    'E_SES': {1: '상', 2: '중상', 3: '중', 4: '중하', 5: '하'},
    'M_STR': {1: '대단히 많이 느낌', 2: '많이 느낌', 3: '조금 느낌', 4: '별로 느끼지 않음', 5: '전혀 느끼지 않음'},
    'F_BR': {1: '주 0일', 2: '주 1일', 3: '주 2일', 4: '주 3일', 5: '주 4일', 6: '주 5일', 7: '주 6일', 8: '주 7일 (매일)'},
    'E_S_RCRD': {1: '상 (1)', 2: '중상 (2)', 3: '중 (3)', 4: '중하 (4)', 5: '하 (5)'}
}

var_names_kr = {
    'SEX': '성별 (SEX)',
    'GRADE': '학년 (GRADE)',
    'E_SES': '가정 경제 상태 (E_SES)',
    'M_STR': '평상시 스트레스 인지 (M_STR)',
    'F_BR': '최근 7일간 아침식사 횟수 (F_BR)',
    'E_S_RCRD': '주관적 학업성적 (E_S_RCRD, 결과변수)',
    'PA_SWD_S': '주중 학습목적 앉아서 보낸 시간 (분/일, PA_SWD_S)'
}

# (1) 범주형 변수 처리
for var in cat_vars:
    # 헤더 행 추가
    table1_rows.append({
        '변수 및 범주': f"■ {var_names_kr[var]}",
        '전체_n': '', '전체_비가중%': '', '전체_가중%': '',
        '권장군_n': '', '권장군_비가중%': '', '권장군_가중%': '',
        '부족군_n': '', '부족군_비가중%': '', '부족군_가중%': ''
    })

    # 범주 값 (결측 포함)
    categories = sorted([x for x in s4[var].unique() if pd.notna(x)])
    if s4[var].isna().sum() > 0:
        categories.append('Missing')

    for cat in categories:
        row_dict = {}
        if cat == 'Missing':
            cat_name = "결측 (Missing)"
            sub_s4 = s4[s4[var].isna()]
            sub_ref = s4[(s4['EXPOSURE'] == 0.0) & (s4[var].isna())]
            sub_exp = s4[(s4['EXPOSURE'] == 1.0) & (s4[var].isna())]
        else:
            cat_name = labels[var].get(cat, str(cat))
            sub_s4 = s4[s4[var] == cat]
            sub_ref = s4[(s4['EXPOSURE'] == 0.0) & (s4[var] == cat)]
            sub_exp = s4[(s4['EXPOSURE'] == 1.0) & (s4[var] == cat)]

        # 전체
        n_tot_c = len(sub_s4)
        unw_pct_tot = (n_tot_c / n_total) * 100
        w_pct_tot = (sub_s4['W'].sum() / s4['W'].sum()) * 100

        # 권장수면군 (EXPOSURE=0.0)
        n_ref_c = len(sub_ref)
        unw_pct_ref = (n_ref_c / n_ref) * 100
        w_pct_ref = (sub_ref['W'].sum() / s4[s4['EXPOSURE'] == 0.0]['W'].sum()) * 100

        # 수면부족군 (EXPOSURE=1.0)
        n_exp_c = len(sub_exp)
        unw_pct_exp = (n_exp_c / n_exp) * 100
        w_pct_exp = (sub_exp['W'].sum() / s4[s4['EXPOSURE'] == 1.0]['W'].sum()) * 100

        table1_rows.append({
            '변수 및 범주': f"   {cat_name}",
            '전체_n': f"{n_tot_c:,}", '전체_비가중%': f"{unw_pct_tot:.2f}", '전체_가중%': f"{w_pct_tot:.2f}",
            '권장군_n': f"{n_ref_c:,}", '권장군_비가중%': f"{unw_pct_ref:.2f}", '권장군_가중%': f"{w_pct_ref:.2f}",
            '부족군_n': f"{n_exp_c:,}", '부족군_비가중%': f"{unw_pct_exp:.2f}", '부족군_가중%': f"{w_pct_exp:.2f}"
        })

# (2) 연속형 변수 처리 (PA_SWD_S)
for var in cont_vars:
    table1_rows.append({
        '변수 및 범주': f"■ {var_names_kr[var]}",
        '전체_n': '', '전체_비가중%': '', '전체_가중%': '',
        '권장군_n': '', '권장군_비가중%': '', '권장군_가중%': '',
        '부족군_n': '', '부족군_비가중%': '', '부족군_가중%': ''
    })

    # 결측 제외 유효 데이터
    valid_s4 = s4.dropna(subset=[var])
    valid_ref = s4[(s4['EXPOSURE'] == 0.0)].dropna(subset=[var])
    valid_exp = s4[(s4['EXPOSURE'] == 1.0)].dropna(subset=[var])

    # 평균 및 SE 산출
    tot_unw_mean, tot_unw_sd = valid_s4[var].mean(), valid_s4[var].std()
    tot_w_mean, tot_w_se = get_complex_survey_mean_se(valid_s4, var)

    ref_unw_mean, ref_unw_sd = valid_ref[var].mean(), valid_ref[var].std()
    ref_w_mean, ref_w_se = get_complex_survey_mean_se(valid_ref, var)

    exp_unw_mean, exp_unw_sd = valid_exp[var].mean(), valid_exp[var].std()
    exp_w_mean, exp_w_se = get_complex_survey_mean_se(valid_exp, var)

    table1_rows.append({
        '변수 및 범주': "   가중 평균 (SE) / 비가중 평균 (SD)",
        '전체_n': f"{len(valid_s4):,}",
        '전체_비가중%': f"{tot_unw_mean:.1f} ({tot_unw_sd:.1f})",
        '전체_가중%': f"{tot_w_mean:.1f} ({tot_w_se:.2f})",

        '권장군_n': f"{len(valid_ref):,}",
        '권장군_비가중%': f"{ref_unw_mean:.1f} ({ref_unw_sd:.1f})",
        '권장군_가중%': f"{ref_w_mean:.1f} ({ref_w_se:.2f})",

        '부족군_n': f"{len(valid_exp):,}",
        '부족군_비가중%': f"{exp_unw_mean:.1f} ({exp_unw_sd:.1f})",
        '부족군_가중%': f"{exp_w_mean:.1f} ({exp_w_se:.2f})"
    })

    # 결측 행 표시
    n_miss_tot = s4[var].isna().sum()
    n_miss_ref = s4[(s4['EXPOSURE'] == 0.0)][var].isna().sum()
    n_miss_exp = s4[(s4['EXPOSURE'] == 1.0)][var].isna().sum()

    table1_rows.append({
        '변수 및 범주': "   결측 (Missing)",
        '전체_n': f"{n_miss_tot:,}", '전체_비가중%': f"{(n_miss_tot/n_total)*100:.2f}", '전체_가중%': '-',
        '권장군_n': f"{n_miss_ref:,}", '권장군_비가중%': f"{(n_miss_ref/n_ref)*100:.2f}", '권장군_가중%': '-',
        '부족군_n': f"{n_miss_exp:,}", '부족군_비가중%': f"{(n_miss_exp/n_exp)*100:.2f}", '부족군_가중%': '-'
    })

# DataFrame으로 변환
df_table1 = pd.DataFrame(table1_rows)

# 열 멀티인덱스 생성
columns_multi = pd.MultiIndex.from_tuples([
    ('연구 변수 및 범주', '구분'),
    (f'전체 대상자 (N={n_total:,})', 'n'),
    (f'전체 대상자 (N={n_total:,})', '비가중 %'),
    (f'전체 대상자 (N={n_total:,})', '가중 % (SE)'),
    (f'권장 수면군 [8~10시간] (N={n_ref:,})', 'n'),
    (f'권장 수면군 [8~10시간] (N={n_ref:,})', '비가중 %'),
    (f'권장 수면군 [8~10시간] (N={n_ref:,})', '가중 % (SE)'),
    (f'수면 부족군 [<8시간] (N={n_exp:,})', 'n'),
    (f'수면 부족군 [<8시간] (N={n_exp:,})', '비가중 %'),
    (f'수면 부족군 [<8시간] (N={n_exp:,})', '가중 % (SE)')
])

df_table1.columns = columns_multi
display(df_table1)

# ---------------------------------------------------------
# 3. 가중% vs 비가중% 절대격차(|Diff|) Top 3 추출
# ---------------------------------------------------------
diff_list = []
for row in table1_rows:
    v_name = row['변수 및 범주'].strip()
    if not v_name.startswith('■') and 'Missing' not in v_name and row['전체_가중%'] != '-':
        try:
            unw = float(row['전체_비가중%'])
            w = float(row['전체_가중%'])
            diff = abs(w - unw)
            diff_list.append({'변수 및 범주': v_name, '비가중%': unw, '가중%': w, '절대 격차(%p)': round(diff, 3)})
        except:
            pass

df_diff = pd.DataFrame(diff_list).sort_values(by='절대 격차(%p)', ascending=False)

print("\n" + "=" * 60)
print("[분석] 전체 대상자 기준 가중% vs 비가중% 절대격차 Top 3")
print("=" * 60)
display(df_diff.head(3))

Table 1 기본 대상자 수: 전체 N = 44,138 | 권장수면군 N = 23,399 | 수면부족군 N = 20,739



연구 변수 및 범주 전체 대상자 (N=44,138)                 \
                                     구분                 n          비가중 %   
0                            ■ 성별 (SEX)                                    
1                                    남성            22,370          50.68   
2                                    여성            21,768          49.32   
3                          ■ 학년 (GRADE)                                    
4                               중학교 1학년             7,051          15.97   
5                               중학교 2학년             7,374          16.71   
6                               중학교 3학년             7,488          16.96   
7                              고등학교 1학년             7,391          16.75   
8                              고등학교 2학년             7,430          16.83   
9                              고등학교 3학년             7,404          16.77   
10                   ■ 가정 경제 상태 (E_SES)                                    
11                                    상             4,632          10.49   
12                                   중상            13,940          31.58   
13                                    중            20,743          47.00   
14                                   중하             4,049           9.17   
15                                    하               770           1.74   
16                         결측 (Missing)                 4           0.01   
17                ■ 평상시 스트레스 인지 (M_STR)                                    
18                            대단히 많이 느낌             4,911          11.13   
19                                많이 느낌            13,535          30.67   
20                                조금 느낌            18,862          42.73   
21                            별로 느끼지 않음             5,663          12.83   
22                            전혀 느끼지 않음             1,167           2.64   
23              ■ 최근 7일간 아침식사 횟수 (F_BR)                                    
24                                 주 0일            11,530          26.12   
25                                 주 1일             3,518           7.97   
26                                 주 2일             4,178           9.47   
27                                 주 3일             3,340           7.57   
28                                 주 4일             2,644           5.99   
29                                 주 5일             4,835          10.95   
30                                 주 6일             2,785           6.31   
31                            주 7일 (매일)            11,308          25.62   
32          ■ 주관적 학업성적 (E_S_RCRD, 결과변수)                                    
33                                상 (1)             5,290          11.99   
34                               중상 (2)            11,524          26.11   
35                                중 (3)            12,830          29.07   
36                               중하 (4)            10,407          23.58   
37                                하 (5)             4,087           9.26   
38  ■ 주중 학습목적 앉아서 보낸 시간 (분/일, PA_SWD_S)                                    
39             가중 평균 (SE) / 비가중 평균 (SD)            43,565  444.0 (243.9)   
40                         결측 (Missing)               573           1.30   

                 권장 수면군 [8~10시간] (N=23,399)                               \
       가중 % (SE)                          n          비가중 %     가중 % (SE)   
0                                                                          
1          50.92                     12,489          53.37         53.63   
2          49.08                     10,910          46.63         46.37   
3                                                                          
4          15.64                      4,523          19.33         19.06   
5          16.33                      4,383          18.73         18.46   
6          16.72                      4,336          18.53         18.30   
7          16.49                      3,529          15.08         14.98   



[분석] 전체 대상자 기준 가중% vs 비가중% 절대격차 Top 3


,변수 및 범주,비가중%,가중%,절대 격차(%p)
7,고등학교 3학년,16.77,17.82,1.05
9,중상,31.58,32.34,0.76
10,중,47.00,46.36,0.64
